# EDA & QA hasil preprocessing

Notebook ini **hanya membaca** `processed/latest/` (output `python -m anmedsos_prep run`). Semua logika preprocessing ada di `src/anmedsos_prep/`. Output sel di-strip otomatis (nbstripout).

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROCESSED = Path(os.environ.get("ANMEDSOS_PROCESSED", "../processed/latest"))
manifest = json.loads((PROCESSED / "manifest.json").read_text(encoding="utf-8"))
print("run:", manifest["run_id"], "| QA lulus:", manifest["qa_passed"], "| git:", manifest["git"])
print("fingerprint salt X:", manifest["x_salt_fingerprint"])
corpus = pd.read_parquet(PROCESSED / "corpus_all.parquet")
views = {p.stem: pd.read_parquet(p) for p in sorted((PROCESSED / "views").glob("*.parquet"))}
corpus.shape

## Ukuran korpus & view

In [ ]:
pd.DataFrame({name: v.groupby(["platform", "kasus"]).size() for name, v in views.items()}).fillna(0).astype(int)

## Attrition

In [ ]:
pd.read_csv(PROCESSED / "attrition_log.csv")

## Volume harian (v_volume)

Garis abu-abu = batas segmen 9 hari; garis merah = laporan dampak kesehatan Karo (hari ke-26). Volume YouTube = komentar *terambil* (D12), volume X bisa dibobot `coverage_weight`.

In [ ]:
vol = views["v_volume"]
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
for i, platform in enumerate(["x", "youtube"]):
    for j, kasus in enumerate(["sukoharjo", "karo"]):
        ax = axes[i, j]
        g = vol[(vol.platform == platform) & (vol.kasus == kasus)]
        g.day.value_counts().sort_index().reindex(range(45), fill_value=0).plot(ax=ax, label="dokumen")
        if platform == "x":
            g.groupby("day").coverage_weight.sum().reindex(range(45), fill_value=0).plot(
                ax=ax, ls="--", label="tertimbang cakupan")
        for b in range(9, 45, 9):
            ax.axvline(b, color="grey", lw=0.5, ls=":")
        if kasus == "karo":
            ax.axvline(26, color="red", lw=1, label="laporan kesehatan")
        ax.set_title(f"{platform} - {kasus}")
        ax.set_xlabel("hari sejak insiden")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Distribusi per segmen & fase

In [ ]:
pd.crosstab([vol.platform, vol.kasus], vol.segmen, margins=True)

In [ ]:
pd.crosstab([vol.platform, vol.kasus], vol.phase, normalize="index").round(3)

## Kata kunci RQ2 (v_rq2 base = v_stance)

Proporsi per kasus x platform; untuk Karo juga sebelum/sesudah laporan kesehatan. Kolom kanan: proporsi makro per video (YouTube, D15).

In [ ]:
st = views["v_stance"]
kw = [c for c in st.columns if c.startswith("kw_")]
pd.concat({"per_dokumen": st.groupby(["platform", "kasus"])[kw].mean(),
           "makro_per_video": st[st.platform == "youtube"].groupby(["kasus", "video_id"])[kw].mean()
                                 .groupby("kasus").mean().assign(platform="youtube")
                                 .set_index("platform", append=True).swaplevel()}).round(3).T

In [ ]:
karo = st[st.kasus == "karo"]
karo.groupby(["platform", "after_health_report"])[kw].mean().round(3).T

## Token teratas (v_topic, text_topic_nostem)

In [ ]:
tp = views["v_topic"]
top = {f"{p}-{k}": [f"{w} ({n})" for w, n in Counter(" ".join(g.text_topic_nostem).split()).most_common(20)]
       for (p, k), g in tp.groupby(["platform", "kasus"])}
pd.DataFrame(top)

## Kedalaman argumentasi (RQ3, menunggu revisi D10)

In [ ]:
st.groupby(["platform", "kasus"])[["n_tokens", "n_sentences", "connectives_per_100_tokens", "has_question",
                                    "has_evidence_marker"]].mean().round(2)

## Analisis sensitivitas

Contoh: jumlah dokumen view stance tanpa 13 video yang kena batas thread (D12), tanpa akun hiperaktif, dan dengan sub-kontroversi Karo (D14).

In [ ]:
import sys
sys.path.insert(0, "../src")
from anmedsos_prep.config import load_config
from anmedsos_prep.corpus import view

cfg = load_config("../config/pipeline.yaml")
pd.DataFrame({
    "utama": view(corpus, "v_stance", cfg).groupby(["platform", "kasus"]).size(),
    "tanpa_video_capped": view(corpus, "v_stance", cfg, exclude_capped=True).groupby(["platform", "kasus"]).size(),
    "tanpa_hiperaktif": view(corpus, "v_stance", cfg, exclude_hyperactive=True).groupby(["platform", "kasus"]).size(),
    "plus_sub_karo": view(corpus, "v_stance", cfg, include_sub=True).groupby(["platform", "kasus"]).size(),
    "x_konteks_lengkap": view(corpus, "v_stance", cfg, require_context=True).groupby(["platform", "kasus"]).size(),
})